In [ ]:
import time

import numpy as np
from numpy import ndarray
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
np.__version__

### Реализация функций активации.

In [ ]:
# ReLU
def relu(x: ndarray) -> ndarray:
    return np.maximum(0, x)

def drelu(act: ndarray, grad: ndarray) -> ndarray:
    return grad*np.where(act>0, 1, 0)


x = np.array([[10, 5, 0, -5, -10]])     # size (1, 5)
grad = np.ones_like(x)                  # size (1, 5)

act_relu = relu(x)
grad_relu = drelu(act_relu, grad)

print(f"{act_relu = }")
print(f"{grad_relu = }")

In [ ]:
# ELU
def elu(x: ndarray, alpha: float=1) -> ndarray:
    return np.where(x>0, x, alpha * (np.exp(x) - 1))

def delu(act: ndarray, grad: ndarray, alpha: float=1) -> ndarray:
    return grad*np.where(act>0, 1, act + alpha)


x = np.array([[10, 5, 0, -0.5, -5, -8, -13, -17, -100]])     # size (1, 9)
grad = np.ones_like(x)                                       # size (1, 9)

alpha=2
act_elu = elu(x, alpha=alpha)
grad_elu = delu(act_elu, grad, alpha=alpha)

print(f"act_elu = {act_elu}\n")
print(f"grad_elu = {grad_elu}")

In [ ]:
# Swish
def swish(x: ndarray) -> ndarray:
    sigmoid = (1 / (1 + np.exp(-x)))
    return sigmoid, x * sigmoid

def dswish(
        act: ndarray,
        grad: ndarray,
        sigm: ndarray
) -> ndarray:
    return grad*(act + sigm * (1 - act))


x = np.array([[10, 5, 0, -0.5, -0.7, -2, -5, -10, -100]])     # size (1, 9)
grad = np.ones_like(x)                                        # size (1, 9)


sigm, act_swish = swish(x)
grad_swish = dswish(act_swish, grad, sigm)

print(f"act_swish = {act_swish}\n")
print(f"grad_swish = {grad_swish}")

In [ ]:
# Sigmoid
def sigmoid(x: ndarray) -> ndarray:
    return (1 / (1 + np.exp(-x)))

def dsigm(act: ndarray, grad: ndarray) -> ndarray:
    return grad * act * (1 - act)


x = np.array([[10, 5, 2, 0.5, 0, -0.5, -2, -5, -10,]])     # size (1, 9)
grad = np.ones_like(x)                                     # size (1, 9)


act_sigmoid = sigmoid(x)
grad_sigm = dsigm(act_sigmoid, grad)

print(f"act_sigmoid = {act_sigmoid}\n")
print(f"grad_sigmoid = {grad_sigm}")

In [ ]:
# Softmax
def softmax(x: ndarray) -> ndarray:
    max_x = np.max(x, axis=1, keepdims=True)
    exp_x = np.exp(x - max_x)
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

def dsoftmax(act: ndarray, grad: ndarray) -> ndarray:
    eye = np.eye(act.shape[1])
    J = act[:, None, :] * (eye[None] - act[..., None])
    return grad @ J


x = np.array([[10, 5, 2, 0.5, 0, -0.5, -2, -5, -10]])     # size (1, 9)
grad = np.random.randn(*x.shape)                          # size (1, 9)


act_softmax = softmax(x)
grad_softmax = dsoftmax(act_softmax, grad)

print(f"act_softmax = {act_softmax}\n")
print(f"grad_softmax = {grad_softmax}")

### Сравнение скорости работы функций активаций.

In [ ]:
def work_time_func(func, n_iter: int = 1000) -> float:
    start = time.perf_counter()
    for _ in range(n_iter):
        func()
    end = time.perf_counter()
    return end - start

def measure_activation_times(
        func_act,
        batch_size: int = 1,
        vector_size: int = 10,
        n_iter: int = 1000
) -> None:
    x = np.random.randn(batch_size, vector_size)
    grad = np.ones((batch_size, vector_size))

    for name, (act, grad_act) in func_act.items():
        act_time = []
        grad_time = []
        for _ in range(100):
            act_time.append(work_time_func(lambda: act(x), n_iter))

            if name == 'Swish':
                sigm, result = act(x)
                grad_time.append(
                    work_time_func(lambda: grad_act(result, grad, sigm), n_iter)
                )
            else:
                result = act(x)
                grad_time.append(
                    work_time_func(lambda: grad_act(result, grad), n_iter)
                )

        print(f"{name}\n\tact_time {np.mean(act_time).round(4)} сек.\n\tgrad_time {np.mean(grad_time).round(4)} сек.")

In [ ]:
func_act = {'ReLU': (relu, drelu), 'ELU': (elu, delu), 'Swish': (swish, dswish)}
measure_activation_times(
    func_act,
    batch_size = 16,
    vector_size = 512,
    n_iter=1000
)

### Затухающие градиенты.

In [ ]:
def func_act(x, f_act):
    return f_act(x)

def func_grad(act, grad, f_grad):
    return f_grad(act, grad)

def plot(name, data):
    
    fig, ax = plt.subplots(1, 2, figsize=(12, 6))
    for i, (mode, bef_aft) in enumerate(data.items()):
        if mode == 'forward':
            labels = [f'{name}_{i+1}' for i in range(len(bef_aft['до активации']))]
            x = np.arange(len(labels))
            width = 0.4
            ax[i].bar(x - width/2, bef_aft['до активации'], width, label='до '+ name, color='g')
            ax[i].bar(x + width/2, bef_aft['после активации'], width, label='после '+ name, color='r')
        else:
            labels = [f'{name}_{i+1}' for i in range(len(bef_aft['до активации']))]
            x = np.arange(len(labels))
            width = 0.4
            ax[i].bar(x - width/2, bef_aft['после активации'], width, label='после '+ name, color='g')
            ax[i].bar(x + width/2, bef_aft['до активации'], width, label='до '+ name, color='r')

        ax[i].set_xlabel('Функция активации')
        ax[i].set_ylabel('Значение')
        ax[i].set_title(mode)
        ax[i].set_xticks(x)
        ax[i].set_xticklabels(labels)
        ax[i].legend()

    plt.show()

In [ ]:
# Первый линейный слой
W1 = np.random.randn(256, 128)
b1 = np.random.randn(1, 128)

# Второй линейный слой
W2 = np.random.randn(128, 64)
b2 = np.random.randn(1, 64)

# Третий линейный слой
W3 = np.random.randn(64, 32)
b3 = np.random.randn(1, 32)

# Четвёртый линейный слой
W4 = np.random.randn(32, 16)
b4 = np.random.randn(1, 16)

# Пятый линейный слой
W5 = np.random.randn(16, 8)
b5 = np.random.randn(1, 8)

In [ ]:
x = np.random.randn(16, 256)    # size(bs, 256)
dict_act = {'ReLU': (relu, drelu), 'ELU': (elu, delu), 'Sigmoid': (sigmoid, dsigm)}

for name, (f_act, f_grad) in dict_act.items():
    before_act = []
    after_act = []
    before_act_grad = []
    after_act_grad = []

    # Прямой проход \ Forward.
    out1 = x@W1 + b1
    before_act.append(out1.mean())

    act1 = func_act(out1, f_act)
    after_act.append(act1.mean())

    out2 = act1@W2 + b2
    before_act.append(out2.mean())

    act2 = func_act(out2, f_act)
    after_act.append(act2.mean())

    out3 = act2@W3 + b3
    before_act.append(out3.mean())

    act3 = func_act(out3, f_act)
    after_act.append(act3.mean())

    out4 = act3@W4 + b4
    before_act.append(out4.mean())

    act4 = func_act(out4, f_act)
    after_act.append(act4.mean())

    out5 = act4@W5 + b5
    before_act.append(out5.mean())

    act5 = func_act(out5, f_act)
    after_act.append(act5.mean())

    # Обратный проход \ Backward.
    grad = np.ones_like(act5)
    before_act_grad.append(np.abs(grad).mean())

    dout5 = func_grad(act5, grad, f_grad)
    after_act_grad.append(np.abs(dout5).mean())

    dact4 = dout5 @ W5.T
    before_act_grad.append(np.abs(dact4).mean())

    dout4 = func_grad(act4, dact4, f_grad)
    after_act_grad.append(np.abs(dout4).mean())

    dact3 = dout4 @ W4.T
    before_act_grad.append(np.abs(dact3).mean())

    dout3 = func_grad(act3, dact3, f_grad)
    after_act_grad.append(np.abs(dout3).mean())

    dact2 = dout3 @ W3.T
    before_act_grad.append(np.abs(dact2).mean())

    dout2 = func_grad(act2, dact2, f_grad)
    after_act_grad.append(np.abs(dout2).mean())

    dact1 = dout2 @ W2.T
    before_act_grad.append(np.abs(dact1).mean())

    dout1 = func_grad(act1, dact1, f_grad)
    after_act_grad.append(np.abs(dout1).mean())

    data = {
        'forward': {
            'до активации': before_act,
            'после активации': after_act
        },
        'backward': {
            'до активации': before_act_grad[::-1],
            'после активации': after_act_grad[::-1]
        }
    }

    plot(name, data)

### Наша первая нейронная сеть.

In [ ]:
# Первый линейный слой
W1 = np.random.randn(784, 128)
b1 = np.random.randn(1, 128)

# Второй линейный слой
W2 = np.random.randn(128, 2)
b2 = np.random.randn(1, 2)


In [ ]:
x = np.random.randn(16, 784)    # size(bs, 784)

# Прямой проход \ Forward.
out1 = x@W1 + b1        # size(bs, 128)
act1 = relu(out1)       # size(bs, 128)

out2 = act1@W2 + b2     # size(bs, 2)

# Обратный проход \ Backward.
grad = np.ones_like(out2)

db2 = np.sum(grad, axis=0, keepdims=True)
dW2 = act1.T @ grad

dact1 = grad @ W2.T

dout1 = drelu(act1, dact1)

db1 = np.sum(dout1, axis=0, keepdims=True)
dW1 = x.T @ dout1



In [ ]:
# Первый линейный слой
W1 = np.random.randn(784, 128)
b1 = np.random.randn(1, 128)

# Второй линейный слой
W2 = np.random.randn(128, 10)
b2 = np.random.randn(1, 10)

In [ ]:
# Прямой проход \ Forward.
out1 = x@W1 + b1
act1 = relu(out1)

out2 = act1@W2 + b2
out = softmax(out2)

# Обратный проход \ Backward.
grad = np.random.randn(*out.shape)

dout2 = dsoftmax(out, grad)

db2 = np.sum(dout2, axis=0, keepdims=True)
dW2 = act1.T @ dout2

dact1 = dout2 @ W2.T

dout1 = drelu(act1, dact1)

db1 = np.sum(dout1, axis=0, keepdims=True)
dW1 = x.T @ dout1